In [ ]:
R_cell                  = None
parameter_median_filter = None
parameter_grey_opening  = None
parameter_grey_closing  = None
nnc                     = None
file_path_NNC           = None
file_path_R             = None
rho_delta               = None
n_DM                    = None
m_DM                    = None
d_xyz                   = None

In [ ]:
%run ./2___Smoothing___Functions.ipynb

---
---
---

### Open data

In [ ]:
with open(file_path_NNC+"grid.pk", 'rb') as f: grid = pkl.load(f)

### DTFE: median, minmax and gray opening/closing filtering

If we are not using the DTFE method, the grid is has too many cells for median, minmax and gray opening/closing filtering.

In [ ]:
if nnc != 0:
    
    # small, structure-preserving median filter
    grid = median_filter(grid, size=parameter_median_filter, mode="wrap")

    # opening: erosion followed by dilation   |   closing: dilation followed by erosion
    # removes small bright residuals/spikes   |   fills small dark gaps/pinholes
    origin_opening = -1 if parameter_grey_opening == 2 else 0
    origin_closing = -1 if parameter_grey_closing == 2 else 0
    grid = grey_closing(grey_opening(grid, size=parameter_grey_opening, origin=origin_opening, mode="wrap"), size=parameter_grey_closing, origin=origin_closing, mode="wrap")

### FFT & Smoothing & Inverse FFT

Worry not! The Gaussian filtering does work over the edges!

In [ ]:
grid_fft = ifftn(fourier_gaussian(fftn(grid), sigma=R_cell)).real

In [ ]:
del grid; gc.collect()

###  Norm

In [ ]:
grid_fft *= n_DM / np.sum(grid_fft)

### Counts -> Mass -> Density

In [ ]:
if rho_delta == "rho": grid_fft *= m_DM / d_xyz**3

### Relative Density

In [ ]:
mean_grid_fft = np.mean(grid_fft)

In [ ]:
if rho_delta == "delta": grid_fft = grid_fft/mean_grid_fft

## Truncation

Since we wish to keep the logarithmic scale finite, we must truncate the grid's values after a certain point.

We consider $\delta = 0.01$ to be a practical minima considering the maxima is $\delta \sim 10^3$.

This process is of great importance in the leveling section, if the log method is used.

This way, we preserve all of the usable information from the Gaussian smoothened frame, since values much smaller simply indicate an empty cell with a far away non-empty cell.

In [ ]:
# Since we used the relative density going forward, we skip this step.
if rho_delta == "delta": grid_fft[grid_fft < 0.01               ] = 0.01
else:                    grid_fft[grid_fft < 0.001*mean_grid_fft] = 0.001*mean_grid_fft

---

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'wb') as f: pkl.dump(grid_fft, f)

---
---
---